# Migración de assets Rocket a Databricks

Este notebook ejecuta paso a paso el flujo público de `py2rocket`: consulta el proyecto, sincroniza todos los workflow assets y sus versiones, y genera notebooks Databricks Source conservando la jerarquía original.

In [ ]:
# Parámetros cargados desde .env (celda compatible con Papermill)
import os
from dotenv import load_dotenv

load_dotenv()

def env_bool(name, default=False):
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}

ROCKET_URL = os.getenv("ROCKET_API_HOST") or os.getenv("ROCKET_URL")
ROCKET_TOKEN = os.getenv("ROCKET_AUTH_COOKIE")
PROJECT = os.getenv("PY2ROCKET_MIGRATION_PROJECT", "")
GROUP_PATH = os.getenv("PY2ROCKET_MIGRATION_GROUP_PATH", "")
OUTPUT_DIR = os.getenv("PY2ROCKET_MIGRATION_OUTPUT", "migracion_databricks")
UNITY_CATALOG_MAPPING = os.getenv("PY2ROCKET_UNITY_CATALOG_MAP") or None
FORCE = env_bool("PY2ROCKET_MIGRATION_FORCE")
VERIFY_SSL = env_bool("ROCKET_VERIFY_SSL", True)
TEMPLATE_REPLACEMENT_ENABLED = env_bool("PY2ROCKET_TEMPLATE_REPLACEMENT", True)
TEMPLATE_NODES = [name.strip() for name in os.getenv("PY2ROCKET_TEMPLATE_NODES", "Parametros,tri_punto_control,tri_registrar_fin,tri_registrar_inicio,sql_rangos_fechas,tri_resumen_ejecucion,pys_notificaciones_ini_tpl,pys_notificaciones_fin_tpl").split(",") if name.strip()]
TEMPLATE_PARAMETER_NODE = os.getenv("PY2ROCKET_TEMPLATE_PARAMETER_NODE", "Parametros")
TEMPLATE_TABLE_FIELD = os.getenv("PY2ROCKET_TEMPLATE_TABLE_FIELD", "tablaUbicacion")
TEMPLATE_OUTPUT_NAME = os.getenv("PY2ROCKET_TEMPLATE_OUTPUT_NAME", "Save_Migrated_Table")
TEMPLATE_SAVE_MODE = os.getenv("PY2ROCKET_TEMPLATE_SAVE_MODE", "Overwrite")
TEMPLATE_SOURCE_NODE = os.getenv("PY2ROCKET_TEMPLATE_SOURCE_NODE") or None

## 1. Preparar la librería y las rutas

In [ ]:
from pathlib import Path

from py2rocket import get_projects
from examples.databricks_migration_demo import (
    build_group_path,
    convert_workflows,
    find_workflows,
    select_project,
    sync_project,
)

template_replacement = {
    "enabled": TEMPLATE_REPLACEMENT_ENABLED,
    "nodes": TEMPLATE_NODES,
    "parameter_node": TEMPLATE_PARAMETER_NODE,
    "table_field": TEMPLATE_TABLE_FIELD,
    "output_name": TEMPLATE_OUTPUT_NAME,
    "save_mode": TEMPLATE_SAVE_MODE,
    "source_node": TEMPLATE_SOURCE_NODE,
}
migration_dir = Path(OUTPUT_DIR).expanduser()
download_dir = migration_dir / "rocket"
notebooks_dir = migration_dir / "databricks"
print(f"Descargas: {download_dir}")
print(f"Notebooks: {notebooks_dir}")

## 2. Consultar los proyectos de Rocket

In [ ]:
projects_result = get_projects(
    rocket_url=ROCKET_URL,
    api_token=ROCKET_TOKEN,
    verify_ssl=VERIFY_SSL,
)
if projects_result.get("status") != "success":
    raise RuntimeError(projects_result.get("message"))

projects = projects_result.get("projects", [])
print(f"Proyectos encontrados: {len(projects)}")
for index, item in enumerate(projects, start=1):
    print(f"{index:>3}. {item.get('name')} ({item.get('normalizedName')})")

## 3. Seleccionar el proyecto y resolver el grupo

In [ ]:
project = select_project(projects, PROJECT or None)
group_path = build_group_path(project, GROUP_PATH)
print(f"Proyecto: {project.get('name')}")
print(f"Ruta Rocket: {group_path}")

## 4. Descargar y convertir los assets al DSL de py2rocket

Esta celda usa `py2rocket sync`, que incluye subgrupos y todas las versiones de cada workflow asset.

In [ ]:
sync_project(
    group_path,
    download_dir,
    rocket_url=ROCKET_URL,
    api_token=ROCKET_TOKEN,
    verify_ssl=VERIFY_SSL,
    force=FORCE,
)
print("Sincronización finalizada")

## 5. Revisar los workflows descargados

In [ ]:
workflow_files = list(find_workflows(download_dir))
print(f"Workflows/versiones listos para convertir: {len(workflow_files)}")
for workflow_file in workflow_files:
    print(f"- {workflow_file.relative_to(download_dir)}")

## 6. Generar los notebooks Databricks Source

In [ ]:
converted, errors = convert_workflows(
    download_dir,
    notebooks_dir,
    UNITY_CATALOG_MAPPING,
    template_replacement,
)
print(f"Notebooks generados: {len(converted)}")
print(f"Conversiones con error: {len(errors)}")

## 7. Resumen de la migración

In [ ]:
print(f"Proyecto: {project.get('name')} ({project.get('normalizedName')})")
print(f"Grupo migrado: {group_path}")
print(f"DSL descargados: {len(workflow_files)}")
print(f"Notebooks generados: {len(converted)}")
print(f"Salida Databricks: {notebooks_dir.resolve()}")
if errors:
    print("\nErrores:")
    for workflow_file, message in errors:
        print(f"- {workflow_file}: {message}")